In [ ]:
# import libraries
import plotly.express as px
import polars as pl
from pathlib import Path

# load dataset
file_path = Path("population_cities.csv")
df = pl.read_csv(file_path)
print(df.head())
print(df.describe())
print(df.null_count())

# cast it from str to int
df = df.with_columns(
    pl.col("Population")
    .str.replace_all(",", "")   # remove commas
    .cast(pl.Int64, strict=False)
)

#upload cleaner dataset version
df.write_csv("cleaned_population_cities.csv")
print(df.describe())


# Aggregation
df_grouped = (
    df.group_by(["City", "Country", "Continent"])
    .agg(pl.sum("Population").alias("total_pop"))
)

# sort top 10
df_top10 = (
    df_grouped.sort("total_pop", descending=True)
    .head(10)
)

# change it to pandas for plotly
top10_pd = df_top10.to_pandas()
df_pd = df.to_pandas()

# visualization
fig1 = px.bar(
    top10_pd,
    x="City",
    y="total_pop",
    color="Country",
    text_auto=".3s",
    title="Top 10 Cities by Population"
)
fig1.show()

fig2 = px.histogram(
    df_pd,
    x="Population",
     color="Continent",
    barmode="group",
    nbins=15, title="Distribution of City Populations"
)
fig2.show()

fig3 = px.treemap(
     top10_pd,
           path=["Country", "City"],
           values="total_pop",
           title="Hierarchical View of Population by Country and City"
)
fig3.show()